**NEURAL NETWORK FOR MNIST DATASET**(From Scratch using Numpy):

Here , It is also similiar as  Multi Classification neural network.

It has also the same process like previous one the only thing here is to convert the image into vector (3D to 2D) then remaining calculation remains same.

The flow of a Neural Network training:(Similar)

DATA  ->  SPLITING(Features,Target)  ->  SCALING DATA  ->  WEIGHTS AND BIAS  ->  FORWARD PROPAGATION  ->  LOSS  ->  BACKWARD PROPAGATION  ->  UPDATE WEIGHTS AND BIAS  ->  **TEST** MODEL

In [347]:
!pip install numpy scikit-learn

Here , we take the MNIST dataset from sklearn.

The data consist of 60000 samples and image size(28*28) .[X(60000,28,28)]. It is dataset hand written numbers images. The target is to  find the handwritten number using model trained on mnist data.


classes are from  0 - 9


In [348]:
import numpy as np
from keras.datasets import mnist

# Load dataset
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Check shapes
print(X_train.shape)  # (60000, 28, 28)
print(y_train.shape)  # (60000,)

(60000, 28, 28)
(60000,)


Here , in this step we have converted 3D array to 2D array . an neural network can't take a 3D input . so we flatten the values .

(60000,28,28) -->> (60000,784)

In [349]:
X_train = X_train.reshape(-1, 784)
X_test  = X_test.reshape(-1, 784)

We have normalized the values . On scaling the computation makes simple and more accurate  with out scaling .

One hot encoding is to label the classes .

In [350]:
X_train = X_train / 255.0
X_test  = X_test / 255.0



def one_hot_encode(y, num_classes):
    return np.eye(num_classes)[y]

num_classes = 10

y_train = one_hot_encode(y_train, num_classes)
y_test  = one_hot_encode(y_test, num_classes)



 n_x -> Sample size 60000 rows


 n_h1 -> Hidden layer1 Neurons

 n_h2 -> HIdden layer2 Neurons

 n_y -> Output Layer  Neurons (num_classes)

 Weights & Bias:

 w1 and b1 ->  for Input Layer


 w2 and b2 -> for Hidden Layer1

 w3 and b3 -> for HIdden Layer2

In [351]:
def intial_parameters(n_x,n_h1,n_h2,n_y):
  w1 = np.random.randn(n_x,n_h1)*0.01
  b1 = np.zeros((1,n_h1))

  w2 = np.random.randn(n_h1,n_h2)*0.01
  b2 = np.zeros((1,n_h2))

  w3 = np.random.randn(n_h2,n_y)*0.01
  b3 = np.zeros((1,n_y))


  return w1,b1,w2,b2,w3,b3


Activation Functions:
 these mattes more in a neural network training it is introduce to learn complex patterns in the data . Introdues non-linearity.

 For multi classification we use softmax activation function

 Softmax : output neuron
 Relu : Hidden Layers

In [352]:
def softmax(z):
  exp_z = np.exp(z)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def Relu(z):
  return np.maximum(0,z)


Forward pass of the neural network here where we train the neural network .
  z=w⋅x+b (mathematical representation)

  z1 -> a1=Relu(z1) -> z2 -> a2=Relu(z2) -> z3 -> a3=softmax(z3)

In [353]:
def forward_propagation(x,w1,b1,w2,b2,w3,b3):
  z1 = np.dot(x,w1)+b1
  a1 = Relu(z1)

  z2 = np.dot(a1,w2)+b2
  a2 = Relu(z2)

  z3 = np.dot(a2,w3)+b3
  a3 = softmax(z3)

  return z1, a1, z2, a2, z3, a3

For Multi Classificaton we use Categorical Cross Entropy loss function to calculate the loss . The loss function shows us how much error the model makes so based on that we can update the weights and bias .

In [354]:
def loss_function(y,y_pred):
  m=y.shape[0]
  loss = -np.sum(y*np.log(y_pred + 1e-8))/m
  return loss

Backward Propagation : This works on the concept Gradient Descent . It calculate the deriavative of the loss and updates the weights and bias .

NOTE:

->The output layer gradient is propostional to how much prediction differs from true label.

-> input*error=gradient

Here the extra content is deriavative of relu function because of we have 2 hidden layers we ned to also calculate gradients for hidden layers also.


In [355]:

def relu_deriavative(z):
  return (z>0).astype(float)


def backward_propagation(x, y, a1, a2, a3, w2, w3, z1, z2):
    m = x.shape[0]

    dz3 = a3 - y
    dw3 = np.dot(a2.T, dz3)
    db3 = np.sum(dz3, axis=0, keepdims=True)/m

    da2 = np.dot(dz3, w3.T)
    dz2 = da2*relu_deriavative(z2)
    dw2 = (1/m)*np.dot(a1.T,dz2)
    db2 = np.sum(dz2, axis=0, keepdims=True)/m

    da1 = np.dot(dz2, w2.T)
    dz1 = da1*relu_deriavative(z1)
    dw1 = (1/m)*np.dot(x.T, dz1)
    db1 = np.sum(dz1, axis=0, keepdims=True)/m

    return dw1, db1, dw2, db2, dw3, db3



Learning rate & Updating of weights and bias:

 -> learning rate is for how much step size the model moving for optimal solution.

 ->it updates the model parameters during the backpropagation.

 ->small learning rate -> better solution


In [356]:
def upgrade_parameters(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, learning_rate):

    w1 = w1 - learning_rate*dw1
    b1 = b1 - learning_rate*db1

    w2 = w2 - learning_rate*dw2
    b2 = b2 - learning_rate*db2

    w3 = w3 - learning_rate*dw3
    b3 = b3 - learning_rate*db3

    return w1, b1, w2, b2, w3, b3

Here , is the training loop starts we train the model for number itereations so that the model learns patterns more effective and decreses the loss function.

We update the weights and bias for number of epochs and uses the best model .

In [357]:
def train(x,y,n_h1,n_h2,epochs,learning_rate):
  n_x=x.shape[1]
  n_y=10

  w1,b1,w2,b2,w3,b3 = intial_parameters(n_x,n_h1,n_h2,n_y)

  for i in range(epochs):
    z1,a1,z2,a2,z3,a3 = forward_propagation(x,w1,b1,w2,b2,w3,b3)
    loss = loss_function(y,a3)
    dw1, db1, dw2, db2 ,dw3, db3 = backward_propagation(x,y, a1, a2, a3, w2, w3, z1,z2)
    w1, b1, w2, b2 , w3, b3 = upgrade_parameters(w1, b1, w2, b2, w3, b3, dw1, db1, dw2, db2, dw3, db3, learning_rate)
    if i % 10 == 0:
        print("Epoch: ", i, "Loss: ", loss)

  return w1,b1,w2,b2,w3,b3


Predict the output hand written digits belong to .

In [358]:
def predict(x,w1,b1,w2,b2,w3,b3):
  z1,a1,z2,a2,z3,a3 = forward_propagation(x,w1,b1,w2,b2,w3,b3)
  return np.argmax(a3, axis=1)

Test Accuracy of the model

-> We observe the model on training the for number of iteration decreses the loss function and increase the acuuracy

In [359]:
np.random.seed(42)

Test on train data set

In [360]:
w1,b1,w2,b2 ,w3,b3= train(X_train,y_train, n_h1=128,n_h2=64, epochs=100,learning_rate=0.01)
y_pred = predict(X_train,w1,b1,w2,b2,w3,b3)
y_pred = one_hot_encode(y_pred,num_classes=10)
acuuracy = np.mean(y_pred == y_train)
print(acuuracy)

Epoch:  0 Loss:  2.3026406675433586
Epoch:  10 Loss:  2.1104864935420258
Epoch:  20 Loss:  1.1313838633175337
Epoch:  30 Loss:  1.6574184991408316
Epoch:  40 Loss:  1.4365320030844855
Epoch:  50 Loss:  1.3065148983275552
Epoch:  60 Loss:  0.7563015709515931
Epoch:  70 Loss:  0.49352669388907494
Epoch:  80 Loss:  0.47752786967930066
Epoch:  90 Loss:  0.42174109031547496
0.9718666666666667


On further we can use optimizers(adam) and can get more acurate output prediction from the model .

-> optimizers are similiar to gradients descent but on using them they are more accurate then the  gradients

Test in test dataset

In [362]:
w1,b1,w2,b2 ,w3,b3= train(X_train,y_train, n_h1=128,n_h2=64, epochs=100,learning_rate=0.01)
y_pred = predict(X_test,w1,b1,w2,b2,w3,b3)
y_pred = one_hot_encode(y_pred,num_classes=10)
acuuracy = np.mean(y_pred == y_test)
print(acuuracy)

Epoch:  0 Loss:  2.3025761743951736
Epoch:  10 Loss:  2.130268803677369
Epoch:  20 Loss:  1.145195151472649
Epoch:  30 Loss:  2.1695980403637978
Epoch:  40 Loss:  1.8129586238264386
Epoch:  50 Loss:  1.470406949542711
Epoch:  60 Loss:  1.2984816991324473
Epoch:  70 Loss:  0.8124279355397765
Epoch:  80 Loss:  0.7361682391106092
Epoch:  90 Loss:  0.5160411540820558
0.9667


In [ ]:
print(y_train.shape)
print(y_test.shape)